# Hit the context window's limits

This notebook goes with the [Context window](https://bradleyboehmke.github.io/ai-engineering-notes/dictionary/context-window.html) dictionary entry. It sends small requests to Claude and shows the ways a request can end:

1. **Count before you send.** Ask the API how many input tokens a prompt uses (free).
2. **Cut off vs finished.** Send the same prompt with a tiny `max_tokens` and a roomy one, and read the stop reason.
3. **Too long (optional).** Send more input than a model's window holds and read the error.

**You need your own Anthropic API key.** Get one from the [Claude Console](https://platform.claude.com/). Steps 1 and 2 cost well under a cent on Claude Haiku 5.5. Step 3 is off by default: it sends about 0.5 MB that the API rejects before generating anything, and even if that input were billed in full it would be about $0.21.

## 1. Install the library

In [ ]:
%pip install --quiet anthropic

## 2. Add your API key

Never paste your key into a cell, because notebooks get shared. In Colab, store it as a secret:

1. Click the **key icon** (Secrets) in the left sidebar.
2. Click **Add new secret**, name it `ANTHROPIC_API_KEY`, and paste your key as the value.
3. Turn on **Notebook access** for this notebook.

If you're not in Colab, or haven't added the secret, the cell below asks for the key instead. Your typing is hidden, and the key lives only in this session's memory.

In [ ]:
import os
from getpass import getpass

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("Using the ANTHROPIC_API_KEY secret.")
except Exception:
    if not os.environ.get("ANTHROPIC_API_KEY"):
        os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key: ")
    print("Using the key from this session.")

## 3. The code

This is the same code as [`limits_api.py`](https://github.com/bradleyboehmke/ai-engineering-notes/blob/main/learning/llm-mechanics/experiments/limits_api.py), split into functions so the cells below can reuse it.

- `count` asks the token-counting endpoint how many input tokens a prompt uses. It's free, and it's an estimate.
- `send` makes one request and returns what came back: the stop reason, the token usage, and the reply text.

Thinking is switched off, so `max_tokens` only has to cover the visible reply. With thinking on, thinking tokens come out of the same `max_tokens` budget.

In [ ]:
import anthropic

client = anthropic.Anthropic()
MODEL = "claude-haiku-5-5"

def count(prompt, model=MODEL):
    """Input tokens this prompt would use, before you send it (free)."""
    messages = [{"role": "user", "content": prompt}]
    return client.messages.count_tokens(model=model, messages=messages).input_tokens

def send(prompt, max_tokens, model=MODEL):
    """One request. Returns the stop reason, token usage, and reply text."""
    r = client.messages.create(
        model=model,
        max_tokens=max_tokens,
        thinking={"type": "disabled"},   # thinking tokens would also count toward max_tokens
        messages=[{"role": "user", "content": prompt}],
    )
    text = next((b.text for b in r.content if b.type == "text"), "")
    return {
        "max_tokens": max_tokens,
        "stop_reason": r.stop_reason,
        "input_tokens": r.usage.input_tokens,
        "output_tokens": r.usage.output_tokens,
        "text": text,
    }

## 4. Count before you send

Enter a prompt and run the cell. The count covers the whole request, not just your text, so it's a little more than the words alone. Compare it with the `input_tokens` the real calls report in step 5: the count is an estimate, so the two can differ by a few tokens.

In [ ]:
prompt = "In about 150 words, explain how to clean a messy sales dataset in Python."  # @param {type:"string"}

print(f"count_tokens: {count(prompt)} input tokens")

## 5. Cut off vs finished

This sends the prompt from step 4 twice. **Before you run it**, write down what you expect each stop reason to be, and how many output tokens the roomy call will use.

A cut-off reply is not an error. You get a normal response with half an answer, and the only signal is `stop_reason`. You're also billed for every output token, finished or not.

In [ ]:
tight_max_tokens = 30  # @param {type:"integer"}
roomy_max_tokens = 1024  # @param {type:"integer"}

import pandas as pd

runs = [send(prompt, tight_max_tokens), send(prompt, roomy_max_tokens)]
table = pd.DataFrame(runs).set_index("max_tokens")
table["ends with"] = table.pop("text").str[-60:]
table

The chart shows how much of each call's `max_tokens` the reply used. A full bar with `stop_reason: max_tokens` means the reply was cut off. A partial bar with `end_turn` means the model stopped on its own: `max_tokens` is a ceiling, not a target length.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 1.0 + 0.6 * len(runs)))
labels = [f"max_tokens={r['max_tokens']:,}" for r in runs]
used = [r["output_tokens"] / r["max_tokens"] for r in runs]
ax.barh(labels, [1] * len(runs), color="#efece0")
ax.barh(labels, used, color=["#8a4620" if r["stop_reason"] == "max_tokens" else "#5b5a57" for r in runs])
for i, r in enumerate(runs):
    ax.text(min(used[i], 1) + 0.01, i, f"{r['output_tokens']:,} tokens · {r['stop_reason']}", va="center", fontsize=9)
ax.set_xlim(0, 1.45)
ax.set_xticks([0, 0.5, 1])
ax.set_xticklabels(["0%", "50%", "100% of max_tokens"])
ax.invert_yaxis()
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
ax.set_title("How much of max_tokens each reply used", loc="left", fontsize=11)
plt.tight_layout()
plt.show()

## 6. Too long (optional)

Tick the box to run it. It asks the Models API for Claude Haiku 4.5's context window (200K tokens, cheaper to overflow than a 1M model), builds a filler prompt about 5% bigger, counts it, and sends it.

**Before you run it**, predict: which HTTP status do you get, how many output tokens are generated, and does the API quietly trim your prompt to fit?

In [ ]:
run_overflow = False  # @param {type:"boolean"}

import math

if run_overflow:
    SMALL = "claude-haiku-4-5"
    window = client.models.retrieve(SMALL).max_input_tokens
    print(f"{SMALL} context window: {window:,} tokens")

    chunk = "The quarterly report lists sales by region and product. " * 1000
    big = chunk * math.ceil(1.05 * window / count(chunk, SMALL))
    try:
        print(f"count_tokens: {count(big, SMALL):,} input tokens (over by design)")
    except anthropic.BadRequestError as e:
        print(f"count_tokens rejected it too: {e.message}")

    try:
        client.messages.create(
            model=SMALL,
            max_tokens=30,
            messages=[{"role": "user", "content": big + "\n\nSummarize the above."}],
        )
        print("Accepted (unexpected)")
    except anthropic.BadRequestError as e:      # 400: nothing was generated
        print(f"HTTP {e.status_code}: {e.message}")
    except anthropic.RateLimitError as e:       # 429: a rate limit hit first
        print(f"HTTP {e.status_code} (a rate limit, not the window): {e.message}")
else:
    print("Skipped. Tick run_overflow and run this cell to try it.")

## What to look for

- **The count is an estimate.** `count_tokens` and the reported `input_tokens` can differ by a few tokens.
- **A cut-off reply looks normal.** The only sign it's incomplete is `stop_reason: max_tokens`. Code that doesn't check it passes half an answer along.
- **`max_tokens` is a ceiling.** The roomy call stopped well short of it. Asking for "about 150 words" in the prompt is a request, while `max_tokens` is the limit that's enforced.
- **Too long fails loudly.** A prompt over the window gets HTTP 400 and nothing is generated. The API doesn't trim your input to make it fit.
- **Your numbers will differ a little.** Replies vary from run to run, so output token counts change. The stop reasons shouldn't.